# V12 Phase 1U — multi-speed HA ownership diagnostic

This notebook reads the deterministic Phase-1U output pack. It is consumed-development evidence only and grants no action authority.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

repo = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'output').is_dir() and (path / 'research' / 'v12').is_dir())
pack = repo / 'output' / 'v12_phase1u_multispeed_ha_ownership_20260926_a'
assert pack.is_dir(), pack
cards = pd.read_csv(pack / 'V12_PHASE1U_POLICY_SCORECARDS.csv').set_index('policy')
states = pd.read_csv(pack / 'V12_PHASE1U_STATE_SCORECARDS.csv')
transitions = pd.read_csv(pack / 'V12_PHASE1U_K1_TRANSITION_SCORECARDS.csv')
summary = json.loads((pack / 'V12_PHASE1U_SUMMARY.json').read_text(encoding='utf-8'))
summary['status'], summary['action_authority']

In [ ]:
comparison = cards[['children', 'hard_sl_children', 'positive_children', 'positive_ge3r', 'positive_ge5r', 'win_rate', 'max_stop_streak', 'weighted_R']].copy()
comparison['hard_sl_saved_vs_control'] = cards.loc['V10_CONTROL', 'hard_sl_children'] - comparison['hard_sl_children']
comparison['positive_lost_vs_control'] = cards.loc['V10_CONTROL', 'positive_children'] - comparison['positive_children']
comparison

In [ ]:
selected_k1 = states[(states.population == 'MT5_SELECTED') & (states.stage == 'k1')].set_index('ownership_state')
selected_k1[['children', 'hard_sl_children', 'hard_sl_rate', 'positive_children', 'positive_ge3r', 'positive_ge5r', 'weighted_R']].sort_values('hard_sl_rate')

In [ ]:
selected_transition = transitions[transitions.population == 'MT5_SELECTED'].set_index('transition_signature')
plot_frame = selected_transition[['hard_sl_children', 'positive_children']].sort_values('hard_sl_children', ascending=False)
ax = plot_frame.plot(kind='bar', figsize=(10, 4), color=['#c94c4c', '#3b82b6'])
ax.set_title('Selected k1 outcomes by causal STD/SLOW transition')
ax.set_ylabel('Child count')
ax.set_xlabel('prior STD/SLOW alignment → current alignment to new FAST direction')
plt.tight_layout()
plt.show()